# Brain state statistics (K-medoids, K = 5)

This notebook performs subject-level statistical analyses on brain state features derived from k-medoids clustering of wSMI connectivity, inspired by the approach in Della Bella et al.

The key steps are:

1. Load subject×condition features:
   - state occupancies \(p_i\) for \(K = 5\)
   - weighted entropy (WE)
   - occupancy entropy \(H(p)\) and its normalized version \(H_{\text{occ\_norm}}\).
2. Define high-entropy and low-entropy state groups based on state entropies \(H_i\).
3. Compute descriptive statistics (mean, SEM, 95% CI) for each condition.
4. Perform within-subject statistical tests across conditions for:
   - WE
   - \(H_{\text{occ\_norm}}\)
   - high-entropy occupancy \(p_\text{high}\)
   - low-entropy occupancy \(p_\text{low}\)
5. Visualise brain state occupancy profiles across states and conditions.


In [ ]:
from pathlib import Path  # For robust path handling

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # For higher-level plotting
from scipy import stats  # For statistical tests

# Plotting style configuration
sns.set(style="whitegrid", context="talk")  # Use a clean, publication-style theme
plt.rcParams["figure.figsize"] = (10, 6)  # Set a default figure size

# Paths (notebook is assumed to live in notebooks/ at repo root)
FEATURES_CSV = "../data/processed/wsmi/events_tmin-0p2_tmax15_familiar_medical_resting_nice_csd/states_features_k_medoids/features_subject_condition_k5.csv"  # Path to subject×condition features
STATE_ENTROPIES_PATH = "../data/processed/wsmi/events_tmin-0p2_tmax15_familiar_medical_resting_nice_csd/states_features_k_medoids/state_entropies_k5.npy"  # Path to state entropies H_i

# Load features CSV
df = pd.read_csv(FEATURES_CSV)  # Load subject×condition features into a DataFrame

# Basic sanity printout
print("Shape of features DataFrame:", df.shape)  # Show number of rows and columns
print("Columns:", df.columns.tolist())  # List all column names
print("Unique subjects:", df["subject"].nunique())  # Count unique subjects
print("Conditions and counts:")  # Announce condition counts
print(df["condition_name"].value_counts())  # Show how many rows per condition

# Ensure subject and condition_name are treated consistently as strings
df["subject"] = df["subject"].astype(str)  # Cast subject IDs to string
df["condition_name"] = df["condition_name"].astype(str)  # Cast condition names to string

# Inspect the first few rows
df.head()  # Display top rows for inspection


Shape of features DataFrame: (225, 12)
Columns: ['subject', 'condition', 'p_0', 'p_1', 'p_2', 'p_3', 'p_4', 'WE', 'H_occ', 'H_occ_norm', 'n_windows', 'condition_name']
Unique subjects: 75
Conditions and counts:
condition_name
Familiar voice    75
Medical staff     75
Resting           75
Name: count, dtype: int64


,subject,condition,p_0,p_1,p_2,p_3,p_4,WE,H_occ,H_occ_norm,n_windows,condition_name
0,0,0,0.0,0.0,0.600000,0.400000,0.0,2.118537,0.673012,0.418166,10,Familiar voice
1,0,1,0.0,0.0,0.200000,0.800000,0.0,2.132007,0.500402,0.310918,10,Medical staff
2,0,2,0.0,0.0,0.666667,0.333333,0.0,2.116292,0.636514,0.395488,3,Resting
3,1,0,0.0,0.0,0.200000,0.500000,0.3,2.157336,1.029653,0.639759,10,Familiar voice
4,1,1,0.0,0.0,0.200000,0.200000,0.6,2.182665,0.950271,0.590436,10,Medical staff


In [ ]:
## State entropies and derived occupancy measures

In this section, state entropies \(H_i\) are loaded and used to define:

- **High-entropy states**: the two states with the highest \(H_i\).
- **Low-entropy states**: the two states with the lowest \(H_i\).

For each subject×condition, two aggregate occupancy measures are then computed:

- \(p_\text{high}\): summed occupancy over high-entropy states.
- \(p_\text{low}\): summed occupancy over low-entropy states.

These quantities mirror the idea of contrasting more complex vs. more stereotyped brain states.
